In [ ]:
import pandas as pd

input_file = "/home/user-kp/anugreha/human/DGIdb/druggable_genes_all.csv"
output_file = "druggable_genes.csv"
df = pd.read_csv(input_file)
df_no_duplicates = df.drop_duplicates(subset="Gene", keep="first")
df_no_duplicates.to_csv(output_file, index=False)
print(f"Removed duplicates. The cleaned file has {len(df_no_duplicates)} rows.")

In [ ]:
#which genes are druggable in TS

import pandas as pd

def find_gene_intersection(file1, file2, output_file, gene_column1='Gene', gene_column2='Gene'):
    genes1 = set(pd.read_csv(file1)[gene_column1])
    genes2 = set(pd.read_csv(file2)[gene_column2])
    common_genes = genes1.intersection(genes2)
    result_df = pd.DataFrame(list(common_genes), columns=['Gene'])
    result_df.to_csv(output_file, index=False)
    
    print(f"Found {len(common_genes)} genes in common")
    print(f"Results saved to {output_file}")
    
    return len(common_genes)

if __name__ == "__main__":
    FILE1 = "/home/user-kp/anugreha/human/TS_gene_symbols.csv"
    FILE2 = "/home/user-kp/anugreha/human/druggable_genes.csv"
    OUTPUT_FILE = "TS_druggable.csv"
    
    find_gene_intersection(
        FILE1,
        FILE2,
        OUTPUT_FILE,
        gene_column1='Gene',  
        gene_column2='Gene'   
    )

In [ ]:
#filtering the druggable gene pairs in TS_[cell type]

import h5py
import pandas as pd
import numpy as np
from tqdm import tqdm

def filter_matrix_and_save_csv(h5_file, gene_ids_file, output_csv, chunk_size=1000):
    print("Reading gene list...")
    filter_genes = pd.read_csv(gene_ids_file)['Gene'].tolist()
    original_genes = pd.read_csv("/home/user-kp/anugreha/human/TS_gene_symbols.csv")['Gene'].tolist()
    

    gene_to_idx = {gene: idx for idx, gene in enumerate(original_genes)}
    
    keep_indices = []
    for gene in filter_genes:
        if gene in gene_to_idx:
            keep_indices.append(gene_to_idx[gene])
    
    keep_indices = sorted(keep_indices)  # Sort indices
    old_to_new = {old_idx: new_idx for new_idx, old_idx in enumerate(keep_indices)}
    
    ordered_genes = [gene for gene in filter_genes if gene in gene_to_idx]
    
    print(f"Found {len(keep_indices)} genes to extract")
    
    filtered_df = pd.DataFrame(index=ordered_genes, columns=ordered_genes)
    
    with h5py.File(h5_file, 'r') as f:
        for i in tqdm(range(0, len(keep_indices), chunk_size), desc="Processing matrix"):

            chunk_indices = keep_indices[i:i + chunk_size]
            if not chunk_indices:
                continue
        
            chunk_data = f['final_matrix'][chunk_indices][:, keep_indices]
            chunk_genes = [ordered_genes[i:i + chunk_size]]
            for row_idx, gene in enumerate(chunk_genes[0]):
                filtered_df.loc[gene, ordered_genes] = chunk_data[row_idx]
    
    print("Saving to CSV...")
    filtered_df.to_csv(output_csv)
    print(f"Saved filtered matrix to {output_csv}")
    
    return filtered_df

if __name__ == "__main__":
    H5_FILE = "/home/user-kp/anugreha/human/hepatocyte/final_matrix_hepatocyte.h5"
    GENE_IDS_FILE = "/home/user-kp/anugreha/human/DGIdb/TS_druggable.csv"
    OUTPUT_CSV = "filtered_TS_druggable.csv"
    
    filter_matrix_and_save_csv(H5_FILE, GENE_IDS_FILE, OUTPUT_CSV)

In [ ]:
def save_zero_interaction_pairs(filtered_matrix, output_csv):
    long_format = filtered_matrix.stack().reset_index()
    long_format.columns = ['Gene_A', 'Gene_B', 'InteractionCount']
    
    zero_interactions = long_format[long_format['InteractionCount'] == 0]
    
    zero_interactions.to_csv(output_csv, index=False)
    print(f"Gene pairs with zero interactions saved to {output_csv}")

if __name__ == "__main__":
    H5_FILE = "/home/user-kp/anugreha/human/hepatocyte/final_matrix_hepatocyte.h5"
    GENE_IDS_FILE = "/home/user-kp/anugreha/human/DGIdb/TS_druggable.csv"
    OUTPUT_CSV = "filtered_TS_druggable.csv"
    ZERO_INTERACTIONS_CSV = "druggable_TS_zero_interactions.csv"

    filtered_matrix = filter_matrix_and_save_csv(H5_FILE, GENE_IDS_FILE, OUTPUT_CSV)
    save_zero_interaction_pairs(filtered_matrix, ZERO_INTERACTIONS_CSV)


In [ ]:
import pandas as pd 
TS_gene_pairs = "/home/user-kp/anugreha/human/gene_symbols/TS_gene_pairs.csv"
druggable_gene_pairs = "/home/user-kp/anugreha/human/DGIdb/druggable_genes.csv"
output_file = "druggable_gene_pairs_TS.csv"

druggable_genes = pd.read_csv(druggable_gene_pairs)
druggable_set = set(druggable_genes['Gene'])

chunk_size = 5000
chunks = pd.read_csv(TS_gene_pairs,chunksize=chunk_size)

with open(output_file,'w') as output:
    for i, chunk in enumerate(chunks):

        chunk["Gene_A_druggable"] = chunk['Gene_A'].isin(druggable_set)
        chunk["Gene_B_druggable"] = chunk['Gene_B'].isin(druggable_set)
        chunk['Druggable_status'] = chunk.apply(lambda row: "Both" if row['Gene_A_druggable'] and row['Gene_B_druggable'] else 'Gene_A' if row['Gene_A_druggable'] else 'Gene_B' if row['Gene_B_druggable'] else 'None', axis=1)
        chunk.to_csv(output,index=False,header=(i==0),mode='a')
        print(f"Processed chunk {i+1}")
print(f"Druggable gene pairs saved to {output_file}")

In [ ]:
import pandas as pd
file_path = "/home/user-kp/anugreha/human/DGIdb/druggable_gene_pairs_TS.csv"
filtered_output = "/home/user-kp/anugreha/human/DGIdb/filtered_druggable_TS_gene_pairs.csv"
chunk_size = 5000
total_rows = sum(1 for _ in open(file_path)) - 1
total_chunks = (total_rows // chunk_size) + (1 if total_rows % chunk_size > 0 else 0)
print(f"Total chunks: {total_chunks}")

chunks = pd.read_csv(file_path, chunksize=chunk_size)
with open(filtered_output, 'w') as output_file:
    for i, chunk in enumerate(chunks):
        filtered_chunk = chunk[
            (chunk['Druggable_status'].notna()) & 
            (chunk['Druggable_status'] != '') & 
            (chunk['Druggable_status'] != 'None')
        ]
        filtered_chunk = chunk[
            (chunk['Gene_A_druggable'] == True) | 
            (chunk['Gene_B_druggable'] == True)
        ]
        filtered_chunk.to_csv(output_file, index=False, header=(i==0), mode='a')
        
        print(f"Processed and saved chunk {i+1} of {total_chunks}")

print("Druggable gene pairs saved successfully")

select only the rows corresponding to druggable genes

In [ ]:
import h5py
import pandas as pd
from tqdm import tqdm
import numpy as np

def filter_matrix_with_druggable_genes(h5_file, gene_ids_file, druggable_genes_file, output_h5_file):
    print("Loading all genes from the gene IDs file...")
    all_genes = pd.read_csv(gene_ids_file)["Gene"].tolist()
    gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}
    
    print("Loading druggable genes from the CSV file...")
    druggable_genes = pd.read_csv(druggable_genes_file)["Gene"].tolist()
    print(f"Total druggable genes in CSV: {len(druggable_genes)}")

    druggable_indices = sorted([
        gene_to_index[gene]
        for gene in druggable_genes
        if gene in gene_to_index
    ])
    print(f"Number of druggable genes found in matrix: {len(druggable_indices)}")
    
    print("Filtering the matrix for druggable genes...")
    with h5py.File(h5_file, "r") as h5:
        matrix = h5["final_matrix"]
        filtered_matrix = matrix[druggable_indices, :]
        filtered_genes = [all_genes[idx] for idx in druggable_indices]
        
        print("Saving the filtered matrix to a new HDF5 file...")
        with h5py.File(output_h5_file, "w") as output_h5:
            output_h5.create_dataset("final_matrix", data=filtered_matrix)
            output_h5.create_dataset("filtered_genes", data=[gene.encode() for gene in filtered_genes])
            
        print(f"Original matrix shape: {matrix.shape}")
        print(f"Filtered matrix shape: {filtered_matrix.shape}")
        print(f"Filtered matrix saved to {output_h5_file}")

h5_file = "/home/user-kp/anugreha/human/matrices/final_matrix_hepatocyte.h5"
gene_ids_file = "/home/user-kp/anugreha/human/gene_symbols/TS_gene_symbols.csv"
druggable_genes_file = "/home/user-kp/anugreha/human/DGIdb/druggable_genes.csv"
output_h5_file = "/home/user-kp/anugreha/human/DGIdb/filtered_matrix_3.0.h5"

filter_matrix_with_druggable_genes(h5_file, gene_ids_file, druggable_genes_file, output_h5_file)

In [ ]:
import h5py
import csv
from tqdm import tqdm

def convert_h5_to_csv(h5_file,gene_ids_file, output_csv):
    with h5py.File(h5_file, "r") as h5:
        matrix = h5["final_matrix"]
        druggable_genes = [gene.decode("utf-8") for gene in h5["filtered_genes"]]
        print("loading genes...")
        all_genes = pd.read_csv(gene_ids_file)["Gene"].tolist()

        print(f"Matrix shape: {matrix.shape}")
        print(f"Number of Druggable Genes (rows): {len(druggable_genes)}")
        print(f"Number of All Genes (columns): {len(all_genes)}")

        with open(output_csv, "w", newline="") as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=["Druggable_Gene", "Gene_B", "Count"])
            writer.writeheader()

            for row_idx in tqdm(range(matrix.shape[0]), desc="Processing rows"):
               
                for col_idx in range(matrix.shape[1]):
                    count = matrix[row_idx, col_idx]
                    if count == 0:  # Only write rows where Count = 0
                        writer.writerow({
                            "Druggable_Gene": druggable_genes[row_idx],
                            "Gene_B": all_genes[col_idx],
                            "Count": count
                        })
                       

    print(f"CSV file created at: {output_csv}")

h5_file = "/home/user-kp/anugreha/human/DGIdb/filtered_matrix_3.0.h5"  
output_csv = "/home/user-kp/anugreha/human/DGIdb/druggable_hepatocyte.csv"  
gene_ids_file = "/home/user-kp/anugreha/human/gene_symbols/TS_gene_symbols.csv"
convert_h5_to_csv(h5_file,gene_ids_file, output_csv)

Removing the already predicted SL pairs

In [ ]:
import pandas as pd
from tqdm import tqdm

def filter_gene_pairs(file1, file2, output_file, chunk_size=100000):
    print("Loading file1...")
    df1 = pd.read_csv(file1)
    print("Processing gene pairs from file1...")
    df1['pair'] = df1.apply(lambda row: tuple(sorted([row['Gene_A'], row['Gene_B']])), axis=1)
    exclude_pairs = set(df1['pair'])
    print(f"Total pairs to exclude: {len(exclude_pairs)}")
    print(f"Processing file2 in chunks (chunk size: {chunk_size})...")
    with pd.read_csv(file2, chunksize=chunk_size) as reader, open(output_file, "w", newline="") as outfile:
        first_chunk = True
        for chunk in tqdm(reader, desc="Processing chunks"):
            chunk['pair'] = chunk.apply(lambda row: tuple(sorted([row['Druggable_Gene'], row['Gene_B']])), axis=1)
            filtered_chunk = chunk[~chunk['pair'].isin(exclude_pairs)].drop(columns=['pair'])
            filtered_chunk.to_csv(outfile, index=False, header=first_chunk)
            first_chunk = False
    print(f"Filtered file saved successfully to {output_file}")

file1 = "/home/user-kp/anugreha/human/SL_predictions/SL_predictions_merged.csv"
file2 = "/home/user-kp/anugreha/human/DGIdb/druggable_hepatocyte.csv"
output_file = "/home/user-kp/anugreha/human/DGIdb/druggable_hepatocyte_final.csv"

filter_gene_pairs(file1, file2, output_file)